# ECE 214B Project 2: Full Final Colab Run

This notebook is a runner only. All scientific/model logic lives in `project/scripts/` and `project/src/`.

Workflow: upload `colab_drop_full_final/` to Google Drive, rename it to `214B_colab_drop_full_final`, keep the dataset zip separately in Drive, edit `DATA_ZIP`, set a GPU runtime, and run all cells. The dataset zip is extracted to local Colab storage before experiments run.


## Setup

Edit the path variables below. `RUN_ASR=True` runs the local Whisper ASR-error experiment. `RUN_LLM=False` and `RUN_LLM_SCENE_GRAPH=False` by default because they use paid API calls.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

DROP_DIR = "/content/drive/MyDrive/214B_colab_drop_full_final"
DATA_ZIP = "/content/drive/MyDrive/path/to/S26_ECE_214B_Mini_Project_2.zip"
EXTRACT_DIR = "/content/214B_data"

RUN_ASR = True
RUN_LLM = False
RUN_LLM_SCENE_GRAPH = False
LLM_MODEL = "gpt-4o-mini"

PROJECT_DIR = f"{DROP_DIR}/project"
OUTPUT_ZIP = f"{DROP_DIR}/outputs/outputs_full_final.zip"
SUMMARY_ZIP = f"{DROP_DIR}/outputs/final_summary_files.zip"

print("DROP_DIR:", DROP_DIR)
print("DATA_ZIP:", DATA_ZIP)
print("EXTRACT_DIR:", EXTRACT_DIR)
print("RUN_ASR:", RUN_ASR)
print("RUN_LLM:", RUN_LLM)
print("RUN_LLM_SCENE_GRAPH:", RUN_LLM_SCENE_GRAPH)


In [ ]:
# Optional LLM setup.
# Prefer Colab secrets or a private runtime environment variable named OPENAI_API_KEY.
# Do not paste API keys into notebooks that may be shared or committed.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

drop_path = Path(DROP_DIR)
project_path = Path(PROJECT_DIR)
data_zip_path = Path(DATA_ZIP)

if not drop_path.exists():
    raise FileNotFoundError(f"DROP_DIR not found: {drop_path}")
if not project_path.exists():
    raise FileNotFoundError(f"PROJECT_DIR not found: {project_path}")
for required in [project_path / "scripts", project_path / "src", project_path / "requirements.txt"]:
    if not required.exists():
        raise FileNotFoundError(f"Missing required project item: {required}")
if not data_zip_path.exists():
    raise FileNotFoundError(f"DATA_ZIP not found. Edit DATA_ZIP near the top of the notebook: {data_zip_path}")
if (RUN_LLM or RUN_LLM_SCENE_GRAPH) and not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("LLM run requested but OPENAI_API_KEY is not set. Use the optional key cell or Colab secrets.")

Path(f"{DROP_DIR}/outputs").mkdir(parents=True, exist_ok=True)
print("Drive/drop validation passed.")


## Dataset Extraction

The dataset zip is extracted to local Colab storage so experiments do not stream WAV files from Drive.


In [ ]:
extract_path = Path(EXTRACT_DIR)
if extract_path.exists():
    shutil.rmtree(extract_path)
extract_path.mkdir(parents=True, exist_ok=True)

print(f"Extracting {DATA_ZIP} -> {EXTRACT_DIR}")
with zipfile.ZipFile(DATA_ZIP, "r") as zf:
    zf.extractall(EXTRACT_DIR)

def find_dataset_dir(root: Path) -> Path:
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]
    for candidate in candidates:
        if all((candidate / name).exists() for name in ["splits", "sessions", "wavs"]):
            return candidate
    raise FileNotFoundError(f"Could not find extracted dataset folder containing splits/, sessions/, wavs/ under {root}")

DATA_DIR = str(find_dataset_dir(extract_path))
print("DATA_DIR:", DATA_DIR)


In [ ]:
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


In [ ]:
import platform

print("Python:", sys.version)
try:
    import torch
    print("torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("WARNING: No GPU is available. CPU experiments can run, but transformer/acoustic embedding experiments may be slow.")
except Exception as exc:
    print("WARNING: torch is not importable yet:", repr(exc))
print("Platform:", platform.platform())


## Core Baselines


In [ ]:
core_commands = [
    [sys.executable, "scripts/audit_dataset.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_text_baseline.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_descriptor_baselines.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_acoustic_descriptor_baseline.py", "--data_dir", DATA_DIR],
]
for command in core_commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)


## Transformer Baselines

These rerun the final required transformer/acoustic baselines on GPU. Full HuBERT layer sweep is intentionally optional and not run by default.


In [ ]:
transformer_commands = [
    [sys.executable, "scripts/run_roberta_text_baseline.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_hubert_baseline.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_wavlm_baseline.py", "--data_dir", DATA_DIR],
]
for command in transformer_commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)


## Fusion And Clinical Thresholding


In [ ]:
fusion_threshold_commands = [
    [sys.executable, "scripts/run_fusion.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_clinical_thresholding.py", "--data_dir", DATA_DIR],
]
for command in fusion_threshold_commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)


## Phase 2 Interpretability


In [ ]:
phase2_commands = [
    [sys.executable, "scripts/run_linguistic_marker_panel.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/analyze_marker_errors.py", "--data_dir", DATA_DIR],
]
for command in phase2_commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)


## Phase 3 Hybrid/Clinical Analyses


In [ ]:
phase3_commands = [
    [sys.executable, "scripts/run_uncertainty_gated_marker_fusion.py"],
    [sys.executable, "scripts/run_marker_error_corrector.py"],
    [sys.executable, "scripts/run_stacked_tfidf_marker_meta_model.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_marker_stratified_thresholding.py", "--data_dir", DATA_DIR],
    [sys.executable, "scripts/run_age_aware_thresholding.py", "--data_dir", DATA_DIR],
]
for command in phase3_commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)


## ASR-Error Experiment

Controlled by `RUN_ASR`. This uses local/open-source Whisper and caches transcripts.


In [ ]:
if RUN_ASR:
    command = [
        sys.executable,
        "scripts/run_asr_error_features.py",
        "--data_dir",
        DATA_DIR,
        "--out_dir",
        "outputs/asr_error_features",
        "--whisper_model",
        "base",
    ]
    print("+", " ".join(command))
    subprocess.run(command, check=True)
    text_fusion_command = [
        sys.executable,
        "scripts/run_asr_transcript_text_fusion.py",
        "--data_dir",
        DATA_DIR,
    ]
    print("+", " ".join(text_fusion_command))
    subprocess.run(text_fusion_command, check=True)
else:
    print("Skipping ASR-error experiment because RUN_ASR=False")


## LLM Clinical Marker Experiment

Controlled by `RUN_LLM`. This sends only anonymous session IDs and transcript text, never audio, age, sex, or labels.


In [ ]:
if RUN_LLM:
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("RUN_LLM=True but OPENAI_API_KEY is not set.")
    command = [
        sys.executable,
        "scripts/run_llm_clinical_marker_panel.py",
        "--data_dir",
        DATA_DIR,
        "--out_dir",
        "outputs/llm_clinical_marker_panel",
        "--model",
        LLM_MODEL,
    ]
    print("+", " ".join(command))
    subprocess.run(command, check=True)
    strat_command = [
        sys.executable,
        "scripts/run_llm_stratified_thresholding.py",
        "--data_dir",
        DATA_DIR,
    ]
    print("+", " ".join(strat_command))
    subprocess.run(strat_command, check=True)
    fusion_opt_command = [
        sys.executable,
        "scripts/run_llm_fusion_optimization.py",
        "--data_dir",
        DATA_DIR,
    ]
    print("+", " ".join(fusion_opt_command))
    subprocess.run(fusion_opt_command, check=True)
else:
    print("Skipping LLM clinical marker panel because RUN_LLM=False")


## LLM Cookie Theft Scene-Graph Scorer

Controlled by `RUN_LLM_SCENE_GRAPH`. This sends only anonymous session IDs and transcript text, never audio, age, sex, or labels.


In [ ]:
if RUN_LLM_SCENE_GRAPH:
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("RUN_LLM_SCENE_GRAPH=True but OPENAI_API_KEY is not set.")
    scene_graph_command = [
        sys.executable,
        "scripts/run_llm_cookie_theft_scene_graph.py",
        "--data_dir",
        DATA_DIR,
        "--out_dir",
        "outputs/llm_cookie_theft_scene_graph",
        "--model",
        LLM_MODEL,
    ]
    print("+", " ".join(scene_graph_command))
    subprocess.run(scene_graph_command, check=True)
else:
    print("Skipping LLM Cookie Theft scene-graph scorer because RUN_LLM_SCENE_GRAPH=False")


### Manual Scene-Graph Batch Helpers

Use these manually if the scene-graph API run disconnects. These cells resume from `outputs/llm_cookie_theft_scene_graph/raw_responses_cache.jsonl` and do not rerun cached sessions.


In [ ]:
# List cache progress; no API calls.
# subprocess.run([
#     sys.executable,
#     "scripts/run_llm_cookie_theft_scene_graph.py",
#     "--data_dir", DATA_DIR,
#     "--out_dir", "outputs/llm_cookie_theft_scene_graph",
#     "--model", LLM_MODEL,
#     "--list_progress",
# ], check=True)

# Run the next 50 missing sessions; resumes from cache.
# subprocess.run([
#     sys.executable,
#     "scripts/run_llm_cookie_theft_scene_graph.py",
#     "--data_dir", DATA_DIR,
#     "--out_dir", "outputs/llm_cookie_theft_scene_graph",
#     "--model", LLM_MODEL,
#     "--only_missing",
#     "--batch_size", "50",
# ], check=True)

# Run all remaining missing sessions; resumes from cache.
# subprocess.run([
#     sys.executable,
#     "scripts/run_llm_cookie_theft_scene_graph.py",
#     "--data_dir", DATA_DIR,
#     "--out_dir", "outputs/llm_cookie_theft_scene_graph",
#     "--model", LLM_MODEL,
#     "--only_missing",
# ], check=True)


## Final Result Collection And Packaging


In [ ]:
subprocess.run([sys.executable, "scripts/collect_required_results.py"], check=True)


In [ ]:
output_zip = Path(OUTPUT_ZIP)
summary_zip = Path(SUMMARY_ZIP)
output_zip.parent.mkdir(parents=True, exist_ok=True)
for path in [output_zip, summary_zip]:
    if path.exists():
        path.unlink()

shutil.make_archive(str(output_zip.with_suffix("")), "zip", root_dir=PROJECT_DIR, base_dir="outputs")

reports_dir = Path(PROJECT_DIR) / "reports"
if reports_dir.exists():
    with zipfile.ZipFile(output_zip, "a", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in reports_dir.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(PROJECT_DIR))

summary_files = [
    "outputs/required_results_summary/results_table.md",
    "outputs/required_results_summary/summary.md",
    "outputs/fusion/results.md",
    "outputs/clinical_thresholding/results.md",
    "outputs/linguistic_marker_panel/results.md",
    "outputs/linguistic_marker_panel/error_analysis/marker_error_analysis_summary.md",
    "outputs/uncertainty_gated_marker_fusion/results.md",
    "outputs/marker_error_corrector/results.md",
    "outputs/stacked_tfidf_marker_meta_model/results.md",
    "outputs/marker_stratified_thresholding/results.md",
    "outputs/age_aware_thresholding/results.md",
    "outputs/asr_error_features/results.md",
    "outputs/asr_transcript_text_fusion/results.md",
    "outputs/llm_clinical_marker_panel/results.md",
    "outputs/llm_fusion_optimization/results.md",
    "outputs/llm_cookie_theft_scene_graph/results.md",
]
with zipfile.ZipFile(summary_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for rel in summary_files:
        path = Path(PROJECT_DIR) / rel
        if path.exists():
            zf.write(path, rel)

print("Full output zip:", output_zip)
print("Full output zip size MB:", round(output_zip.stat().st_size / (1024 * 1024), 2))
print("Summary zip:", summary_zip)
print("Summary zip size KB:", round(summary_zip.stat().st_size / 1024, 1))


## Optional Slow HuBERT Layer Sweep

Full HuBERT layer sweep is intentionally not run by default. Uncomment and run manually only if needed.


In [ ]:
# Optional: full HuBERT layer sweep, slow. Not run by default.
# subprocess.run([sys.executable, "scripts/run_hubert_layer_sweep.py", "--data_dir", DATA_DIR], check=True)
